# Hybrid Skill Scoring & ML Analysis (Part 2)

This notebook loads `master_student_data.csv` (generated by Part 1), computes rule-based scores, trains ML models, generates SWOT/recommendations, and creates visual spider charts. Detailed comments included.

In [1]:
# CELL 8 — Part 2: Load master_student_data.csv (analysis & scoring)


import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from math import pi
import logging
import joblib
import warnings
import os  # <-- REQUIRED FIX

from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, mean_squared_error, r2_score


def safe_series(df, col, default=0):
    """Return a Series from df[col], or a Series of default values with proper index length."""
    if col in df.columns:
        return pd.to_numeric(df[col], errors='coerce').fillna(default)
    else:
        return pd.Series([default] * len(df), index=df.index)

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

print("="*80)
print("Hybrid Skill Scoring & ML Analysis (Part 2)")
print("="*80)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")
logging.info("Hybrid Skill Scoring (Part 2) - Starting")

MASTER_OUTPUT = "master_student_data.csv"
if not os.path.exists(MASTER_OUTPUT):
    raise FileNotFoundError("master_student_data.csv not found. Run hybrid_datamerger.ipynb first.")

master_df = pd.read_csv(MASTER_OUTPUT)
logging.info(f"Master loaded: rows={len(master_df)}, cols={len(master_df.columns)}")

# ensure numeric columns are numeric where possible
for col in master_df.columns:
    if master_df[col].dtype == object:
        converted = pd.to_numeric(master_df[col], errors='coerce')
        if converted.notna().sum() > 0:
            master_df[col] = converted

# Fill defaults
num_cols = master_df.select_dtypes(include=[np.number]).columns
master_df[num_cols] = master_df[num_cols].fillna(0)
master_df[master_df.select_dtypes(include=['object']).columns] = master_df.select_dtypes(include=['object']).fillna('')

INFO:root:Hybrid Skill Scoring (Part 2) - Starting
INFO:root:Master loaded: rows=20, cols=20


Hybrid Skill Scoring & ML Analysis (Part 2)


## Rule-based Skill Scoring

Apply the transparent rule-based scoring functions to compute skill-level scores and overall performance index.

In [2]:
# ===========================================================
# CELL 9 — Updated Rule-based Skill Scoring (Corrected & Stable)
# ===========================================================

def to_numeric_safe_local(val, max_value=None):
    try:
        if val is None:
            return 0.0
        if isinstance(val, (int, float, np.number)):
            return float(val)
        s = str(val).strip()
        if s == "" or s.lower() in ['nan', 'none']:
            return 0.0

        # remove labels like %, CGPA, marks
        s = re.sub(r'[%\s]*(cgpa|CGPA|GPA|marks|Marks)\b', '', s)

        nums = re.findall(r'\d+\.?\d*', s)
        if not nums:
            return 0.0
        num = float(nums[0])

        if max_value:
            return float(min(num, max_value))
        return float(num)
    except:
        return 0.0


# ----------------------------------------------------------
# Corrected External Exam Score (Your major issue)
# ----------------------------------------------------------
def calculate_external_exam_score(row):
    score = 0.0

    # EXTERNAL MARKS ARE OUT OF 50 — FIXED
    external_marks = to_numeric_safe_local(row.get('avg_external_marks', 0))
    ext_norm = (external_marks / 50.0) * 100.0         # normalize to %  
    score += (ext_norm * 0.60)                          # 60% weight

    # CGPA (0–10) contributes 40%
    cgpa = to_numeric_safe_local(row.get('avg_cgpa', 0))
    score += (cgpa / 10.0) * 40.0

    # Trend bonus/penalty
    cgpa_trend = str(row.get('cgpa_trend', 'stable')).lower()
    if cgpa_trend == 'improving':
        score += 5
    elif cgpa_trend == 'declining':
        score -= 5

    return float(max(0, min(round(score, 2), 100)))


# ----------------------------------------------------------
# Attendance Discipline Score (Stabilized)
# ----------------------------------------------------------
def calculate_attendance_discipline_score(row):
    attendance = to_numeric_safe_local(row.get('overall_attendance', 0))
    trend = str(row.get('attendance_trend', 'unknown')).lower()

    # Base score
    if attendance >= 90:
        score = 90
    elif attendance >= 80:
        score = 80
    elif attendance >= 70:
        score = 65
    elif attendance >= 60:
        score = 50
    else:
        score = max(25, attendance * 0.7)

    # Trend adjustments
    if trend == 'improving':
        score += 10
    elif trend == 'declining':
        score -= 10
    elif trend == 'stable':
        score += 3

    # Consistency adjustment (soft)
    consistency = to_numeric_safe_local(row.get('attendance_consistency_score', 50))
    score += (consistency / 100.0) * 5

    return float(max(0, min(round(score, 2), 100)))


# ----------------------------------------------------------
# Internal Assessment Score
# ----------------------------------------------------------
def calculate_internal_assessment_score(row):
    avg_iat = to_numeric_safe_local(row.get('avg_iat_score', 0))
    trend = str(row.get('iat_trend', 'stable')).lower()

    score = avg_iat  # already out of 100

    if trend == 'improving':
        score += 5
    elif trend == 'declining':
        score -= 5

    latest_iat = to_numeric_safe_local(row.get('latest_iat_score', 0))
    if latest_iat > avg_iat + 5:
        score += 3

    return float(max(0, min(round(score, 2), 100)))


# ----------------------------------------------------------
# Foundation Score (SSLC + PUC)
# ----------------------------------------------------------
def calculate_foundation_score(row):
    sslc = to_numeric_safe_local(row.get('sslc_percentage', 0))
    puc = to_numeric_safe_local(row.get('puc_percentage', 0))

    score = (sslc * 0.40) + (puc * 0.60)
    return float(max(0, min(round(score, 2), 100)))


# ----------------------------------------------------------
# Holistic Development (Internships + Activities + Skills)
# ----------------------------------------------------------
def calculate_holistic_development_score(row):
    score = 20.0

    # Internship impact
    internship_count = int(to_numeric_safe_local(row.get('internship_count', 0)))
    if internship_count >= 3: score += 30
    elif internship_count == 2: score += 20
    elif internship_count == 1: score += 10

    # Activity diversity
    activity_count = int(to_numeric_safe_local(row.get('activity_count', 0)))
    if activity_count >= 10: score += 25
    elif activity_count >= 5: score += 15
    elif activity_count >= 2: score += 8
    elif activity_count >= 1: score += 5

    # TYL skills
    tyl = int(to_numeric_safe_local(row.get('tyl_skills_tracked', 0)))
    if tyl >= 5: score += 10
    elif tyl >= 3: score += 6
    elif tyl >= 1: score += 3

    return float(max(0, min(round(score, 2), 100)))


# ----------------------------------------------------------
# Consistency Score (Stabilized)
# ----------------------------------------------------------
def calculate_consistency_score(row):
    trends = [
        str(row.get('attendance_trend', '')).lower(),
        str(row.get('iat_trend', '')).lower(),
        str(row.get('cgpa_trend', '')).lower()
    ]

    improving = trends.count('improving')
    declining = trends.count('declining')

    score = 50.0

    if improving >= 2: score += 20
    elif improving == 1: score += 10
    if declining >= 2: score -= 20
    elif declining == 1: score -= 10

    avg_consistency = to_numeric_safe_local(row.get('consistency_index', 50))
    score = (score + avg_consistency) / 2

    return float(max(0, min(round(score, 2), 100)))


# ----------------------------------------------------------
# APPLY ALL SCORING FUNCTIONS
# ----------------------------------------------------------
master_df['external_exam_performance'] = master_df.apply(calculate_external_exam_score, axis=1)
master_df['attendance_discipline'] = master_df.apply(calculate_attendance_discipline_score, axis=1)
master_df['internal_assessment'] = master_df.apply(calculate_internal_assessment_score, axis=1)
master_df['foundation_strength'] = master_df.apply(calculate_foundation_score, axis=1)
master_df['holistic_development'] = master_df.apply(calculate_holistic_development_score, axis=1)
master_df['consistency'] = master_df.apply(calculate_consistency_score, axis=1)

skill_columns = [
    'external_exam_performance', 'attendance_discipline', 'internal_assessment',
    'foundation_strength', 'holistic_development', 'consistency'
]

weights = {
    'external_exam_performance': 0.40,
    'attendance_discipline': 0.25,
    'internal_assessment': 0.15,
    'holistic_development': 0.10,
    'consistency': 0.05,
    'foundation_strength': 0.05
}

master_df['overall_performance_index'] = sum(master_df[col] * weights[col] for col in skill_columns).round(2)
logging.info("Rule-based scoring complete (UPDATED).")


INFO:root:Rule-based scoring complete (UPDATED).


## ML Training & Predictions

Train Risk classifier, CGPA predictor, and Category classifier. Models are saved to `models/`. Detailed logs are printed.

In [3]:
# ================================================================
# CELL 10 — ML MODEL TRAINING (Safe, Normalized, Fully Robust)
# ================================================================

ML_ENABLED = True
TRAIN_ML_MODELS = True
MODEL_PATH = "models"
os.makedirs(MODEL_PATH, exist_ok=True)

logging.info("ML: Starting adaptive, safe training pipeline...")

# ---------------------------------------------------------------
# SAFE COLUMN GETTER (fixes your fillna() crash)
# ---------------------------------------------------------------
def col(df, name, default=0):
    """Always return a SERIES even if column missing, ensuring numeric output."""
    if name in df.columns:
        return pd.to_numeric(df[name], errors='coerce').fillna(default)
    return pd.Series([default] * len(df), index=df.index)

_master = master_df  # alias

# ---------------------------------------------------------------
# NORMALIZATION HELPERS
# ---------------------------------------------------------------
def scale_if_out_of_50(series):
    try:
        s = pd.to_numeric(series, errors='coerce')
        if s.dropna().empty:
            return s.fillna(0)

        mx = s.max()
        if mx <= 50.0:
            return (s.fillna(0) * 2).clip(0, 100)
        return s.fillna(0).clip(0, 100)
    except:
        return pd.Series(0, index=series.index)

# Normalize IAT / External marks
_master['avg_iat_score_norm'] = scale_if_out_of_50(col(_master, 'avg_iat_score'))
_master['latest_iat_score_norm'] = scale_if_out_of_50(col(_master, 'latest_iat_score'))
_master['avg_external_marks_norm'] = scale_if_out_of_50(col(_master, 'avg_external_marks'))

# ---------------------------------------------------------------
# CGPA INFERENCE
# ---------------------------------------------------------------
_master['avg_cgpa'] = col(_master, 'avg_cgpa')
_master['latest_cgpa'] = col(_master, 'latest_cgpa')

cgpa_missing_ratio = (_master['avg_cgpa'] == 0).sum() / len(_master)
if cgpa_missing_ratio > 0.4:
    inferred = (_master['avg_external_marks_norm'] / 10.0).clip(0, 10)
    mask = (_master['avg_cgpa'] == 0)
    _master.loc[mask, 'avg_cgpa'] = inferred[mask]

mask_latest = (_master['latest_cgpa'] == 0)
_master.loc[mask_latest, 'latest_cgpa'] = _master.loc[mask_latest, 'avg_cgpa']

# ---------------------------------------------------------------
# FEATURE LIST
# ---------------------------------------------------------------
full_feature_list = [
    'avg_cgpa','latest_cgpa','first_cgpa','cgpa_volatility','cgpa_improvement_rate',
    'avg_external_marks_norm','avg_iat_score_norm','latest_iat_score_norm','iat_volatility','iat_improvement_rate',
    'overall_attendance','latest_semester_attendance','attendance_volatility','attendance_consistency_score',
    'sslc_percentage','puc_percentage','weighted_foundation',
    'internship_count','activity_count','tyl_skills_tracked','engagement_score','profile_diversity',
    'academic_momentum','consistency_index','overall_volatility','academic_health_index',
    'fail_count','pass_rate','semesters_count','subjects_attended','iat_subjects_count'
]

available_features = [c for c in full_feature_list if c in _master.columns]
logging.info(f"Available ML Features ({len(available_features)}): {available_features}")

X = _master[available_features].fillna(0) if available_features else pd.DataFrame(index=_master.index)

# ---------------------------------------------------------------
# TRAIN MODELS
# ---------------------------------------------------------------
if ML_ENABLED and TRAIN_ML_MODELS:

    # -----------------------------------------------------------
    # 1) RISK CLASSIFICATION MODEL
    # -----------------------------------------------------------
    logging.info("Training Risk Classification Model (safe)...")

    risk_rule = (
        (col(_master, 'overall_performance_index') < 50) |
        (col(_master, 'avg_cgpa') < 5.0) |
        (col(_master, 'overall_attendance') < 75) |
        (col(_master, 'fail_count') > 2)
    )

    _master['at_risk'] = risk_rule.astype(int)

    y_risk = _master['at_risk']
    class_counts = y_risk.value_counts().to_dict()
    logging.info(f"Risk class counts: {class_counts}")

    stratify_param = y_risk if (y_risk.value_counts() >= 2).all() else None

    X_risk = X if X.shape[1] > 0 else pd.DataFrame({
        'avg_cgpa': col(_master, 'avg_cgpa'),
        'overall_attendance': col(_master, 'overall_attendance')
    })

    X_train, X_test, y_train, y_test = train_test_split(
        X_risk, y_risk, test_size=0.2, random_state=42, stratify=stratify_param
    )

    scaler_risk = StandardScaler()
    X_train_s = scaler_risk.fit_transform(X_train)
    X_test_s = scaler_risk.transform(X_test)

    rf_risk = RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_split=8,
        class_weight='balanced', random_state=42
    )
    rf_risk.fit(X_train_s, y_train)
    y_pred = rf_risk.predict(X_test_s)
    logging.info("Risk Model Report:\n" +
                 classification_report(y_test, y_pred, zero_division=0))

    _master['ml_risk_probability'] = rf_risk.predict_proba(scaler_risk.transform(X_risk))[:,1] * 100
    _master['ml_risk_prediction'] = rf_risk.predict(scaler_risk.transform(X_risk))

    joblib.dump(rf_risk, os.path.join(MODEL_PATH, "risk_classifier.pkl"))
    joblib.dump(scaler_risk, os.path.join(MODEL_PATH, "risk_scaler.pkl"))

    # -----------------------------------------------------------
    # 2) CGPA PREDICTION MODEL
    # -----------------------------------------------------------
    logging.info("Training CGPA Prediction Model...")

    if 'semesters_count' not in _master.columns:
        _master['semesters_count'] = 1

    students_hist = _master[_master['semesters_count'] >= 2]

    if len(students_hist) >= 50:
        X_cgpa = students_hist[available_features].fillna(0)
        y_cgpa = students_hist['latest_cgpa']

        X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
            X_cgpa, y_cgpa, test_size=0.2, random_state=42
        )

        scaler_cgpa = StandardScaler()
        X_train_cs = scaler_cgpa.fit_transform(X_train_c)
        X_test_cs = scaler_cgpa.transform(X_test_c)

        gb_cgpa = GradientBoostingRegressor(
            n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42
        )
        gb_cgpa.fit(X_train_cs, y_train_c)

        pred = gb_cgpa.predict(X_test_cs)
        logging.info(f"CGPA RMSE={np.sqrt(mean_squared_error(y_test_c, pred)):.3f}, "
                     f"R2={r2_score(y_test_c, pred):.3f}")

        _master['ml_predicted_next_cgpa'] = gb_cgpa.predict(
            scaler_cgpa.transform(X.fillna(0))
        ).clip(0, 10).round(2)

        joblib.dump(gb_cgpa, os.path.join(MODEL_PATH, "cgpa_predictor.pkl"))
        joblib.dump(scaler_cgpa, os.path.join(MODEL_PATH, "cgpa_scaler.pkl"))

    else:
        _master['ml_predicted_next_cgpa'] = _master['latest_cgpa']

    # -----------------------------------------------------------
    # 3) PERFORMANCE CATEGORY (Balanced)
    # -----------------------------------------------------------
    def categorize_perf(score):
        if score > 75: return 'A - Excellent'
        if score > 60: return 'B - Very Good'
        if score > 45: return 'C - Good'
        if score > 30: return 'D - Needs Improvement'
        return 'E - Critical'

    _master['performance_category'] = _master['overall_performance_index'].apply(
        categorize_perf
    )

    master_df = _master
    logging.info("ML pipeline completed successfully.")

else:
    logging.info("ML disabled; skipping training.")


INFO:root:ML: Starting adaptive, safe training pipeline...
INFO:root:Available ML Features (15): ['avg_cgpa', 'latest_cgpa', 'avg_external_marks_norm', 'avg_iat_score_norm', 'latest_iat_score_norm', 'overall_attendance', 'attendance_consistency_score', 'sslc_percentage', 'puc_percentage', 'internship_count', 'activity_count', 'tyl_skills_tracked', 'consistency_index', 'fail_count', 'pass_rate']
INFO:root:Training Risk Classification Model (safe)...
INFO:root:Risk class counts: {1: 11, 0: 9}
INFO:root:Risk Model Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4

INFO:root:Training CGPA Prediction Model...
INFO:root:ML pipeline completed successfully.


## SWOT & Recommendations

Generate explainable SWOT insights and prioritized recommendations using both rules and ML outputs.

In [4]:
# ============================================================
# CELL 11 — CLEAN, ROBUST SWOT & RECOMMENDATIONS GENERATOR
# ============================================================
def to_numeric_safe(x, default=0.0, max_value=None):
    try:
        if x is None:
            return float(default)
        if isinstance(x, (int, float)):
            val = float(x)
        else:
            s = str(x).strip()
            if s == '' or s.lower() in ['nan', 'none', 'null']:
                return float(default)
            s = s.replace('%', '')
            import re
            m = re.search(r"\d+\.?\d*", s)
            if m:
                val = float(m.group())
            else:
                return float(default)

        if max_value is not None and val > max_value:
            # scale down unrealistic values
            if val > max_value * 5:
                val = val / 10.0

        if pd.isna(val):
            return float(default)
        return val
    except Exception:
        return float(default)

def getnum(row, col, default=0.0):
    """Wrapper around to_numeric_safe with fallback."""
    return to_numeric_safe(row.get(col, default))


def generate_hybrid_swot(row):
    strengths, weaknesses, opportunities, threats = [], [], [], []

    # ---- 1. ACADEMICS / EXTERNAL EXAMS ----
    external_score = getnum(row, 'external_exam_performance')
    avg_cgpa = getnum(row, 'avg_cgpa')
    latest_cgpa = getnum(row, 'latest_cgpa')
    cgpa_trend = str(row.get('cgpa_trend', '')).lower()

    if external_score >= 80:
        strengths.append(f"Excellent external performance (CGPA {avg_cgpa:.2f})")
        if cgpa_trend == 'improving':
            strengths.append("Strong upward CGPA trend")

    if external_score < 50:
        weaknesses.append("Weak external exam performance")
        if cgpa_trend == 'declining':
            weaknesses.append("Declining CGPA trend detected")

    # ---- 2. ATTENDANCE ----
    attendance = getnum(row, 'overall_attendance')
    attendance_discipline = getnum(row, 'attendance_discipline')
    attendance_trend = str(row.get('attendance_trend', '')).lower()

    if attendance_discipline >= 85:
        strengths.append(f"Excellent attendance ({attendance:.1f}%)")

    if attendance_discipline < 75:
        weaknesses.append(f"Low attendance ({attendance:.1f}%)")
        threats.append("Attendance risk — possible detention")

    if attendance_trend == 'declining':
        threats.append("Declining attendance trend")

    # ---- 3. INTERNAL ASSESSMENTS (IAT) ----
    iat_score = getnum(row, 'internal_assessment')
    iat_trend = str(row.get('iat_trend', '')).lower()

    if iat_score >= 70:
        strengths.append("Strong internal assessment performance")
    if iat_score < 50:
        weaknesses.append("Weak internal assessment performance")

    if iat_trend == 'declining':
        threats.append("IAT score trend declining")
    elif iat_trend == 'improving':
        opportunities.append("Improving IAT trend")

    # ---- 4. HOLISTIC DEVELOPMENT ----
    internship_count = int(getnum(row, 'internship_count'))
    activity_count = int(getnum(row, 'activity_count'))
    holistic = getnum(row, 'holistic_development')

    if holistic >= 65:
        strengths.append("Good holistic development profile")
        if internship_count > 0:
            strengths.append(f"Completed {internship_count} internship(s)")
        if activity_count >= 5:
            strengths.append(f"Active in {activity_count} extracurricular activities")
    else:
        if internship_count == 0:
            weaknesses.append("No internship experience")
        if activity_count < 2:
            weaknesses.append("Limited extracurricular participation")

    # ---- 5. CONSISTENCY / MOMENTUM ----
    consistency_index = getnum(row, 'consistency_index')
    academic_momentum = getnum(row, 'academic_momentum')

    if consistency_index >= 70:
        strengths.append("Consistent academic performance")

    if academic_momentum > 0:
        opportunities.append("Positive academic momentum — maintain progress")
    elif academic_momentum < 0:
        threats.append("Negative academic momentum detected")

    # ---- 6. ML RISK INSIGHTS ----
    ml_risk = getnum(row, 'ml_risk_probability')
    predicted_cgpa = getnum(row, 'ml_predicted_next_cgpa')

    if ml_risk > 70:
        threats.append(f"⚠ HIGH ML Risk: {ml_risk:.0f}% probability of poor performance")
    elif ml_risk > 40:
        opportunities.append("Medium ML risk — early intervention recommended")

    if latest_cgpa and predicted_cgpa:
        if predicted_cgpa > latest_cgpa + 0.5:
            opportunities.append(f"Projected CGPA improvement to {predicted_cgpa:.2f}")
        elif predicted_cgpa < latest_cgpa - 0.5:
            threats.append(f"Predicted CGPA drop to {predicted_cgpa:.2f}")

    # ---- 7. Final cleanup ----
    if not strengths:
        strengths.append("Shows potential for improvement with guidance")

    if not weaknesses:
        weaknesses.append("No major weaknesses detected")

    if not opportunities:
        opportunities.append("Can benefit from structured academic planning")

    if not threats:
        threats.append("No immediate threats detected")

    return json.dumps({
        'strengths': strengths,
        'weaknesses': weaknesses,
        'opportunities': opportunities,
        'threats': threats
    })


def generate_hybrid_recommendations(row):
    recs = []
    priority = []

    external_score = getnum(row, 'external_exam_performance')
    attendance = getnum(row, 'overall_attendance')
    holistic = getnum(row, 'holistic_development')
    ml_risk = getnum(row, 'ml_risk_probability')
    predicted_cgpa = getnum(row, 'ml_predicted_next_cgpa')
    latest_cgpa = getnum(row, 'latest_cgpa')

    # --- Priority (RED FLAG) ACTIONS ---
    if ml_risk > 70:
        priority.append("🚨 Schedule immediate meeting with academic advisor")
        priority.append("Enroll in academic support program")

    if external_score < 50:
        priority.append("URGENT: Focus on improving external exam preparation")

    if attendance < 75:
        priority.append("CRITICAL: Improve attendance above 75% immediately")

    # --- Regular improvement recommendations ---
    if external_score < 65:
        recs.append("Practice PYQs weekly to strengthen exam performance")

    if holistic < 50:
        if int(getnum(row, 'internship_count')) == 0:
            recs.append("Apply for internships to gain real-world experience")
        if int(getnum(row, 'activity_count')) < 3:
            recs.append("Participate in at least one technical/cultural club")

    if predicted_cgpa and latest_cgpa:
        if predicted_cgpa < latest_cgpa - 0.3:
            recs.append("Increase study hours — CGPA risk detected")
        elif predicted_cgpa > latest_cgpa + 0.3:
            recs.append("Great potential! Maintain consistency")

    # --- Excellence encouragement ---
    if external_score >= 75 and attendance >= 80:
        recs.append("Consider mentoring juniors or assisting in clubs")

    final = priority + recs
    if not final:
        final = ["Maintain current performance", "Explore leadership opportunities"]

    return json.dumps(final)


# APPLY BOTH
master_df['swot_analysis'] = master_df.apply(generate_hybrid_swot, axis=1)
master_df['recommendations'] = master_df.apply(generate_hybrid_recommendations, axis=1)

logging.info("✓ SWOT + Recommendations generated successfully.")


INFO:root:✓ SWOT + Recommendations generated successfully.


## Save enriched dataset & Spider chart utility

In [5]:
# ================================================================
# CELL 12 — SAVE ENRICHED DATASET & GENERATE SPIDER CHARTS SAFELY
# ================================================================

output_file = "student_skill_analysis_hybrid.csv"
master_df.to_csv(output_file, index=False)
logging.info(f"Enriched dataset saved: {output_file}")

# Ensure directory exists
CHART_DIR = "spider_charts_hybrid"
os.makedirs(CHART_DIR, exist_ok=True)

# ---------------------------------------------------------------
# SAFE GET FUNCTION FOR SPIDER CHART
# ---------------------------------------------------------------
def safe_val(row, col, default=0):
    try:
        return float(row.get(col, default)) if row.get(col, default) not in [None, "", np.nan] else default
    except:
        return default

# ---------------------------------------------------------------
# CLEAN FIXED SPIDER CHART FUNCTION
# ---------------------------------------------------------------
def create_hybrid_spider_chart(student_data, save_path=None):

    # 6 KEY DIMENSIONS (must match your scoring logic)
    categories = [
        "External Exams\n(40%)",
        "Attendance\n(25%)",
        "Internal Test\n(15%)",
        "Foundation\n(5%)",
        "Holistic Dev\n(10%)",
        "Consistency\n(5%)"
    ]

    values = [
        safe_val(student_data, "external_exam_performance", 0),
        safe_val(student_data, "attendance_discipline", 0),
        safe_val(student_data, "internal_assessment", 0),
        safe_val(student_data, "foundation_strength", 0),
        safe_val(student_data, "holistic_development", 0),
        safe_val(student_data, "consistency", 0)
    ]

    N = len(categories)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    values += values[:1]
    angles += angles[:1]

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='polar')

    # Plot
    ax.plot(angles, values, linewidth=2.5, label="Score")
    ax.fill(angles, values, alpha=0.25)

    # Reference target (75)
    ax.plot(angles, [75] * (N + 1), "--", linewidth=1.2, alpha=0.6, label="Target 75")

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=10, fontweight='bold')

    ax.set_ylim(0, 100)
    ax.set_yticks([25, 50, 75, 100])
    ax.grid(True, linestyle="--", alpha=0.7)

    # Title block
    userid = student_data.get("userid", "Unknown")
    perf_index = safe_val(student_data, "overall_performance_index", 0)
    cat = student_data.get("performance_category", "Unknown")

    title = f"Student Performance Profile\nID: {userid} • Score: {perf_index:.1f} • {cat}"

    # ML risk annotation
    if "ml_risk_probability" in student_data:
        risk = safe_val(student_data, "ml_risk_probability", 0)
        if risk > 70:
            title += f"\n⚠️ High Risk ({risk:.0f}%)"
        elif risk > 40:
            title += f"\n⚡ Medium Risk ({risk:.0f}%)"
        else:
            title += f"\n✓ Low Risk ({risk:.0f}%)"

    if "ml_predicted_next_cgpa" in student_data:
        pred = safe_val(student_data, "ml_predicted_next_cgpa", 0)
        title += f" • Predicted CGPA: {pred:.2f}"

    plt.title(title, fontsize=13, fontweight="bold", pad=20)

    # Background color
    if perf_index >= 80:
        fig.patch.set_facecolor("#E8F5E9")
    elif perf_index >= 60:
        fig.patch.set_facecolor("#FFF9C4")
    else:
        fig.patch.set_facecolor("#FFEBEE")

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())

    plt.close(fig)
    return save_path

# ---------------------------------------------------------------
# GENERATE FIRST 20 SPIDER CHARTS SAFELY
# ---------------------------------------------------------------
print("\nGenerating spider charts...\n")
chart_count = min(len(master_df), 20)

for idx in range(chart_count):
    student = master_df.iloc[idx].to_dict()
    userid = student.get("userid", f"student_{idx+1}")
    chart_path = os.path.join(CHART_DIR, f"{userid}_chart.png")

    try:
        create_hybrid_spider_chart(student, save_path=chart_path)
    except Exception as e:
        logging.error(f"Error creating chart for {userid}: {e}")

# ---------------------------------------------------------------
# SUMMARY REPORT
# ---------------------------------------------------------------
print("\n============================================================")
print("             HYBRID SKILL SCORING SUMMARY")
print("============================================================\n")

print(f"Total students analyzed: {len(master_df)}\n")

# Category Distribution
if "performance_category" in master_df.columns:
    print("Performance Distribution:\n")
    print(master_df["performance_category"].value_counts().to_string())
else:
    print("Performance category missing.")

# Top-level metrics
print("\nAverage Overall Performance Index:",
      f"{master_df['overall_performance_index'].mean():.2f}")

if "ml_risk_probability" in master_df.columns:
    high_risk = (master_df["ml_risk_probability"] > 70).sum()
    print(f"High Risk (>70%): {high_risk} students "
          f"({high_risk/len(master_df)*100:.1f}%)")

print("\nOutputs saved:")
print(f" • Enriched CSV: {output_file}")
print(f" • Spider charts directory: {CHART_DIR}/")
print("============================================================")

logging.info("Spider chart generation & summary complete.")


INFO:root:Enriched dataset saved: student_skill_analysis_hybrid.csv



Generating spider charts...



INFO:root:Spider chart generation & summary complete.



             HYBRID SKILL SCORING SUMMARY

Total students analyzed: 20

Performance Distribution:

performance_category
D - Needs Improvement    7
A - Excellent            5
B - Very Good            5
C - Good                 3

Average Overall Performance Index: 60.00
High Risk (>70%): 8 students (40.0%)

Outputs saved:
 • Enriched CSV: student_skill_analysis_hybrid.csv
 • Spider charts directory: spider_charts_hybrid/


## Generate spider charts (sample) and print summary report